# Tool-call verifier / ranker training notebook

This notebook builds and trains a **small ML sidecar** for tool-call guardrails. It is not intended to replace deterministic guardrails. The model classifies or ranks candidate tool calls after cheap deterministic checks have already handled obvious failures.

Target use case:

- deterministic guardrails remain authoritative
- model helps classify/rank semantically suspicious tool calls
- model improves nudge specificity and reduces wasted retry loops
- exported artifacts can run in Rust through ONNX Runtime

Outputs:

- `toolcall_verifier_dataset.jsonl`
- `train.jsonl`, `valid.jsonl`, `test.jsonl`
- fine-tuned classifier checkpoint
- optional ONNX export
- optional Hugging Face Hub upload


## Revision note

This version fixes notebook issues found during smoke and first real runs:

- robust parsing for Glaive-style `system`/`chat` rows
- safe train/valid/test splitting when rare labels have too few examples for stratification
- GPU-aware training defaults for T4/L4/A100 style Colab runtimes
- full-label evaluation so `classification_report` does not fail when the test split lacks some labels
- optional class weighting
- diagnostic vs production label mode
- additional Forge-style synthetic guardrail rows for rare classes such as `missing_prerequisite` and `needs_clarification`

With only Forge synthetic rows, the notebook is a mechanics smoke test. For useful training, confirm that public rows are being parsed and that each label has enough examples.

## 0. Runtime notes

Recommended Colab runtime:

- GPU: T4 or better
- For fast smoke tests, keep `MAX_PER_SOURCE` low
- For real training, increase `MAX_PER_SOURCE`, add Forge traces, and train 2 to 4 epochs

The notebook is defensive because public tool-calling datasets vary in shape.


In [ ]:
#@title Install dependencies
!pip -q install -U \
  "datasets>=2.20.0" \
  "transformers>=4.41.0" \
  "accelerate>=0.30.0" \
  "evaluate>=0.4.2" \
  "scikit-learn>=1.3.0" \
  "huggingface_hub>=0.23.0" \
  "jsonschema>=4.22.0" \
  "optimum[onnxruntime]>=1.20.0" \
  "onnxruntime>=1.18.0"


In [ ]:
#@title Imports, GPU setup, and global config
import os
import re
import gc
import ast
import json
import time
import math
import random
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/content/toolcall-verifier")
DATA_DIR = WORKDIR / "data"
MODEL_DIR = WORKDIR / "model"
ONNX_DIR = WORKDIR / "onnx"
for p in [WORKDIR, DATA_DIR, MODEL_DIR, ONNX_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Start small for a Colab smoke test. Increase after the pipeline is validated.
MAX_PER_SOURCE = 2_000

# Model choices:
# - "distilbert-base-uncased": faster debug runs, weaker accuracy
# - "microsoft/deberta-v3-small": stronger, slower
BASE_MODEL = "microsoft/deberta-v3-small"

# Speed/quality knobs.
# 512 is much faster than 768. Run the token-length diagnostic below before raising it.
MAX_LENGTH = 512
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 1e-5

# Training behavior.
USE_CLASS_WEIGHTS = True
MAX_CLASS_WEIGHT = 5.0
AUTO_FIND_BATCH_SIZE = True
GROUP_BY_LENGTH = True

# Label mode:
# - "diagnostic": full violation taxonomy
# - "production": collapse deterministic failures into one class, leaving semantic labels clearer
LABEL_MODE = "diagnostic"

USE_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if USE_CUDA else "CPU"
USE_BF16 = bool(USE_CUDA and torch.cuda.is_bf16_supported())
USE_FP16 = bool(USE_CUDA and not USE_BF16)

if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Device:", GPU_NAME)
print("USE_CUDA:", USE_CUDA)
print("USE_FP16:", USE_FP16)
print("USE_BF16:", USE_BF16)

if USE_CUDA:
    try:
        !nvidia-smi
    except Exception:
        pass

In [ ]:
#@title Optional: Hugging Face auth for private/gated datasets or upload
# In Colab, add a secret named HF_TOKEN from the left sidebar Secrets tab.
# This cell is safe to skip for public dataset loading.

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Authenticated to Hugging Face.")
else:
    print("No HF_TOKEN found. Public dataset loading should still work.")


## 1. Label schema

The classifier predicts one of these violation classes. Deterministic guardrails should still own hard checks such as malformed JSON, unknown tool names, and required-step enforcement. The ML model is most useful for semantic classes like `wrong_tool_semantic`, `tool_not_needed`, and ambiguous next-tool ranking.


In [ ]:
RAW_LABELS = [
    "valid",
    "wrong_tool_semantic",
    "invalid_args_schema",
    "missing_required_args",
    "unknown_tool",
    "premature_terminal",
    "missing_prerequisite",
    "unsafe_parallel_batch",
    "tool_not_needed",
    "needs_clarification",
    "malformed_tool_call",
]

SEMANTIC_LABEL_MAP = {
    "valid": "valid",
    "wrong_tool_semantic": "wrong_tool_semantic",
    "tool_not_needed": "tool_not_needed",
    "needs_clarification": "needs_clarification",

    # These are normally deterministic Rust guardrail failures.
    # Collapsing them helps train a production sidecar that focuses on semantic ambiguity.
    "invalid_args_schema": "deterministic_invalid",
    "missing_required_args": "deterministic_invalid",
    "unknown_tool": "deterministic_invalid",
    "premature_terminal": "deterministic_invalid",
    "missing_prerequisite": "deterministic_invalid",
    "unsafe_parallel_batch": "deterministic_invalid",
    "malformed_tool_call": "deterministic_invalid",
}

def normalize_label(label: str) -> str:
    if LABEL_MODE == "production":
        return SEMANTIC_LABEL_MAP.get(label, label)
    return label

if LABEL_MODE == "production":
    LABELS = ["valid", "wrong_tool_semantic", "tool_not_needed", "needs_clarification", "deterministic_invalid"]
else:
    LABELS = RAW_LABELS

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

(DATA_DIR / "labels.json").write_text(json.dumps({
    "label_mode": LABEL_MODE,
    "raw_labels": RAW_LABELS,
    "labels": LABELS,
    "semantic_label_map": SEMANTIC_LABEL_MAP,
    "label2id": label2id,
    "id2label": id2label,
}, indent=2))

label2id

## 2. Canonical verifier row format

Each training row is a candidate-call judgment, not a generation target.

This lets the model answer:

> Given the user request, workflow state, available tools, and candidate tool call, how likely is this candidate to be correct?


In [ ]:
@dataclass
class VerifierRow:
    id: str
    source: str
    input_text: str
    label: str
    rank_score: float
    metadata: Dict[str, Any]

def stable_id(*parts: Any) -> str:
    h = hashlib.sha256(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:16]
    return h

def compact_json(obj: Any, max_chars: int = 2500) -> str:
    try:
        s = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except TypeError:
        s = str(obj)
    if len(s) > max_chars:
        return s[:max_chars] + "...<truncated>"
    return s

def serialize_tool(tool: Dict[str, Any]) -> str:
    name = tool.get("name") or tool.get("function", {}).get("name") or "unknown_tool"
    desc = tool.get("description") or tool.get("function", {}).get("description") or ""
    params = tool.get("parameters") or tool.get("function", {}).get("parameters") or {}
    return f"{name}: {desc}\nPARAMETERS: {compact_json(params, 1000)}"

def serialize_state(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
) -> str:
    tool_text = "\n\n".join(serialize_tool(t) for t in tools[:12])
    return f"""USER_REQUEST:
{user_request}

WORKFLOW_STATE:
required_steps={required_steps or []}
completed_steps={completed_steps or []}
pending_steps={pending_steps or []}
terminal_tools={terminal_tools or []}
recent_errors={recent_errors or []}

AVAILABLE_TOOLS:
{tool_text}

CANDIDATE_CALL:
{compact_json(candidate, 2000)}
""".strip()

def make_row(
    source: str,
    label: str,
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    rank_score: float,
    metadata: Optional[Dict[str, Any]] = None,
    required_steps: Optional[List[str]] = None,
    completed_steps: Optional[List[str]] = None,
    pending_steps: Optional[List[str]] = None,
    terminal_tools: Optional[List[str]] = None,
    recent_errors: Optional[List[str]] = None,
) -> VerifierRow:
    input_text = serialize_state(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
        recent_errors=recent_errors,
    )
    rid = stable_id(source, label, user_request, tools, candidate, required_steps, completed_steps, pending_steps, terminal_tools)
    return VerifierRow(
        id=rid,
        source=source,
        input_text=input_text,
        label=label,
        rank_score=float(rank_score),
        metadata=metadata or {},
    )


## 3. Dataset loaders

The loaders below attempt to normalize several public function-calling datasets into a common shape:

```python
{
  "user_request": str,
  "tools": list[dict],
  "gold_calls": list[dict],
  "source": str
}
```

They are intentionally permissive. Inspect samples before large-scale training.


In [ ]:
def try_json_loads(x: Any) -> Any:
    if isinstance(x, (dict, list)):
        return x
    if x is None:
        return None
    if not isinstance(x, str):
        return x
    s = x.strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return ast.literal_eval(s)
    except Exception:
        return x

def get_first_text(record: Dict[str, Any], keys: List[str]) -> str:
    for k in keys:
        v = record.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def balanced_json_objects(text: str, start_pos: int = 0) -> List[str]:
    """Extract balanced {...} JSON-like objects from text."""
    objects = []
    i = start_pos
    while i < len(text):
        if text[i] != "{":
            i += 1
            continue

        start = i
        depth = 0
        in_str = False
        escape = False

        while i < len(text):
            ch = text[i]
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = not in_str
            elif not in_str:
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        objects.append(text[start:i + 1])
                        break
            i += 1
        i += 1
    return objects

def extract_json_after_marker(text: str, marker: str) -> List[Any]:
    out = []
    if not isinstance(text, str) or marker not in text:
        return out
    cursor = 0
    while True:
        idx = text.find(marker, cursor)
        if idx < 0:
            break
        start = idx + len(marker)
        objs = balanced_json_objects(text, start)
        if objs:
            parsed = try_json_loads(objs[0])
            out.append(parsed)
            # Advance past this object to avoid repeated extraction.
            cursor = text.find(objs[0], start) + len(objs[0])
        else:
            cursor = start
    return out

def extract_first_user_from_chat(chat: Any) -> str:
    chat = try_json_loads(chat)
    if isinstance(chat, list):
        for m in chat:
            if isinstance(m, dict) and m.get("from") in ("user", "human"):
                return str(m.get("value", "")).strip()
    if not isinstance(chat, str):
        return ""

    # Glaive format: USER: ... ASSISTANT:
    m = re.search(r"USER:\s*(.*?)(?:\s+ASSISTANT:|<\|endoftext\|>|$)", chat, flags=re.S)
    if m:
        return m.group(1).strip()

    return ""

def extract_messages_text(messages: Any) -> str:
    messages = try_json_loads(messages)
    if isinstance(messages, list):
        parts = []
        for m in messages:
            if not isinstance(m, dict):
                continue
            role = m.get("role", m.get("from", ""))
            content = m.get("content", m.get("value", ""))
            if isinstance(content, list):
                content = " ".join(str(c) for c in content)
            if role in ("user", "human") and content:
                parts.append(str(content))
        return "\n".join(parts).strip()
    return extract_first_user_from_chat(messages)

def normalize_tool_object(t: Any) -> Optional[Dict[str, Any]]:
    t = try_json_loads(t)
    if not isinstance(t, dict):
        return None

    if t.get("type") == "function" and isinstance(t.get("function"), dict):
        f = t["function"]
        return {
            "name": f.get("name", "unknown_tool"),
            "description": f.get("description", ""),
            "parameters": f.get("parameters", {"type": "object", "properties": {}}),
        }

    name = t.get("name") or t.get("tool_name") or t.get("api_name") or t.get("function_name")
    if not name and isinstance(t.get("function"), dict):
        name = t["function"].get("name")
    if not name:
        return None

    params = (
        t.get("parameters")
        or t.get("parameter")
        or t.get("args")
        or t.get("arguments_schema")
        or t.get("input_schema")
        or {"type": "object", "properties": {}}
    )
    params = try_json_loads(params)
    if not isinstance(params, dict):
        params = {"type": "object", "properties": {}}

    return {
        "name": str(name),
        "description": str(t.get("description") or t.get("desc") or ""),
        "parameters": params,
    }

def extract_tools_from_text(text: Any) -> List[Dict[str, Any]]:
    """Parse tools embedded in system/chat text, especially Glaive's `system` field."""
    if not isinstance(text, str):
        return []

    tools = []

    # Glaive system field often contains one or more JSON function definitions.
    for obj_text in balanced_json_objects(text):
        obj = try_json_loads(obj_text)
        if isinstance(obj, list):
            for item in obj:
                tool = normalize_tool_object(item)
                if tool:
                    tools.append(tool)
        else:
            tool = normalize_tool_object(obj)
            if tool:
                tools.append(tool)

    # Deduplicate by name.
    seen = set()
    deduped = []
    for t in tools:
        name = t.get("name")
        if name and name not in seen:
            seen.add(name)
            deduped.append(t)
    return deduped

def extract_tools(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidate_keys = [
        "tools", "functions", "function", "tool", "apis", "api_list",
        "function_list", "available_tools", "function_definition"
    ]
    for k in candidate_keys:
        if k not in record:
            continue
        raw = try_json_loads(record.get(k))
        if isinstance(raw, dict):
            if any(key in raw for key in ["name", "function", "tool_name", "api_name"]):
                maybe = normalize_tool_object(raw)
                return [maybe] if maybe else []
            for nested_key in ["tools", "functions", "apis"]:
                if isinstance(raw.get(nested_key), list):
                    raw = raw[nested_key]
                    break
        if isinstance(raw, list):
            out = []
            for item in raw:
                t = normalize_tool_object(item)
                if t:
                    out.append(t)
            if out:
                return out

    # Glaive-style: tools are embedded in `system`; sometimes also in `chat`.
    for text_key in ["system", "chat", "conversation", "conversations"]:
        out = extract_tools_from_text(record.get(text_key))
        if out:
            return out

    return []

def normalize_call_object(c: Any) -> Optional[Dict[str, Any]]:
    c = try_json_loads(c)
    if isinstance(c, list) and c:
        return normalize_call_object(c[0])
    if not isinstance(c, dict):
        return None

    if isinstance(c.get("function"), dict):
        f = c["function"]
        args = try_json_loads(f.get("arguments", {}))
        if not isinstance(args, dict):
            args = {"_raw": args}
        return {"name": f.get("name", "unknown_tool"), "arguments": args}

    name = c.get("name") or c.get("tool_name") or c.get("function_name") or c.get("api_name")
    args = c.get("arguments", c.get("args", c.get("parameters", c.get("input", {}))))
    args = try_json_loads(args)
    if not isinstance(args, dict):
        args = {"_raw": args}

    if name:
        return {"name": str(name), "arguments": args}

    if isinstance(c.get("tool"), dict):
        return normalize_call_object(c["tool"])

    return None

def parse_toolace_call_text(value: str) -> Optional[Dict[str, Any]]:
    """Parse ToolACE assistant strings like `[GetFutureEvents(page=1)]`."""
    if not isinstance(value, str):
        return None
    m = re.search(r"\[([A-Za-z0-9_ ./-]+)\((.*?)\)\]", value)
    if not m:
        return None
    name = m.group(1).strip()
    raw_args = m.group(2).strip()
    args = {}
    if raw_args:
        # Handles simple key=value pairs. Good enough for hard-negative pretraining.
        for part in re.split(r",\s*(?=[A-Za-z_][A-Za-z0-9_]*\s*=)", raw_args):
            if "=" not in part:
                continue
            k, v = part.split("=", 1)
            k = k.strip()
            v = v.strip()
            try:
                args[k] = ast.literal_eval(v)
            except Exception:
                args[k] = v.strip('"').strip("'")
    return {"name": name, "arguments": args}

def extract_gold_calls(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidate_keys = [
        "tool_calls", "function_call", "function_calls", "answers",
        "answer", "assistant", "calls", "api_calls", "gold_calls"
    ]
    calls = []
    for k in candidate_keys:
        if k not in record:
            continue
        raw = try_json_loads(record.get(k))
        if isinstance(raw, list):
            for item in raw:
                call = normalize_call_object(item)
                if call:
                    calls.append(call)
        else:
            call = normalize_call_object(raw)
            if call:
                calls.append(call)
        if calls:
            return calls

    # Glaive format: ASSISTANT: <functioncall> {"name": ..., "arguments": "..."}
    for text_key in ["chat", "assistant", "output", "conversation"]:
        text_blob = record.get(text_key, "")
        if isinstance(text_blob, str) and "<functioncall>" in text_blob:
            for obj in extract_json_after_marker(text_blob, "<functioncall>"):
                call = normalize_call_object(obj)
                if call:
                    calls.append(call)

    # ToolACE conversation list: assistant values look like `[Function(arg=...)]`
    conv = try_json_loads(record.get("conversations"))
    if isinstance(conv, list):
        for m in conv:
            if isinstance(m, dict) and m.get("from") == "assistant":
                call = parse_toolace_call_text(m.get("value", ""))
                if call:
                    calls.append(call)

    return calls

def normalize_record(record: Dict[str, Any], source: str) -> Optional[Dict[str, Any]]:
    user_request = (
        get_first_text(record, ["query", "question", "instruction", "user", "prompt", "input"])
        or extract_messages_text(record.get("messages"))
        or extract_messages_text(record.get("conversation"))
        or extract_messages_text(record.get("conversations"))
        or extract_first_user_from_chat(record.get("chat"))
    )
    tools = extract_tools(record)
    gold_calls = extract_gold_calls(record)

    # ToolACE does not always expose clean structured tool schemas. If we have a call but no tools,
    # create a minimal schema from the call so it can still contribute ranking examples.
    if not tools and gold_calls:
        tools = [
            {"name": c["name"], "description": "Tool parsed from conversation.", "parameters": {"type": "object", "properties": {}}}
            for c in gold_calls if c.get("name")
        ]

    if not user_request or not tools or not gold_calls:
        return None

    return {
        "source": source,
        "user_request": user_request,
        "tools": tools,
        "gold_calls": gold_calls,
    }

def load_source_dataset(name: str, split: str = "train", max_rows: int = MAX_PER_SOURCE, **kwargs) -> List[Dict[str, Any]]:
    print(f"Loading {name}...")
    try:
        ds = load_dataset(name, split=split, streaming=False, **kwargs)
    except Exception as e:
        print(f"  load_dataset failed for {name}: {type(e).__name__}: {e}")
        return []

    if max_rows:
        ds = ds.select(range(min(max_rows, len(ds))))

    rows = []
    for rec in tqdm(ds, desc=f"normalizing {name}"):
        norm = normalize_record(dict(rec), source=name)
        if norm:
            rows.append(norm)

    print(f"  normalized {len(rows)} usable rows out of {len(ds)}")
    if len(rows) == 0 and len(ds) > 0:
        sample = dict(ds[0])
        print("  No rows parsed. Sample keys:", list(sample.keys()))
        print("  Sample preview:", json.dumps({k: str(v)[:500] for k, v in sample.items()}, indent=2))
    return rows


In [ ]:
#@title Load public function-calling sources
# Keep this list editable. Some datasets change schema or access rules over time.
#
# Notes:
# - Salesforce/xLAM may require accepting dataset terms and HF auth.
# - BFCL explicitly says not to use load_dataset; it is best used later as a JSONL eval source.
# - Glaive is the most reliable public bootstrap source here because its viewer exposes 113k train rows.
PUBLIC_SOURCES = [
    ("glaiveai/glaive-function-calling-v2", {"split": "train"}),
    ("Team-ACE/ToolACE", {"split": "train"}),
    ("Salesforce/xlam-function-calling-60k", {"split": "train"}),
    # Keep BFCL disabled for training ingestion. Use its JSONL files as a separate eval harness.
    # ("gorilla-llm/Berkeley-Function-Calling-Leaderboard", {"split": "train"}),
]

normalized_examples = []
source_counts = {}
for dataset_name, kwargs in PUBLIC_SOURCES:
    rows = load_source_dataset(dataset_name, max_rows=MAX_PER_SOURCE, **kwargs)
    normalized_examples.extend(rows)
    source_counts[dataset_name] = len(rows)

print("Usable rows by source:")
for k, v in source_counts.items():
    print(f"  {k}: {v}")
print("Total normalized examples:", len(normalized_examples))

if normalized_examples:
    print(json.dumps(normalized_examples[0], indent=2)[:4000])
else:
    print("No public rows parsed. The notebook will continue with Forge synthetic rows only, but training will be a smoke test, not a useful model.")


## 4. Forge-specific synthetic traces

Public datasets do not know about your workflow contract concepts:

- required steps
- terminal tools
- respond/final tools
- missing prerequisites
- unsafe terminal+nonterminal batches

So we add Forge-style synthetic rows directly.


In [ ]:
FORGE_WORKFLOWS = [
    {
        "source": "forge_synthetic",
        "user_request": "Generate a sales report from the Q4 2024 dataset.",
        "tools": [
            {
                "name": "fetch_sales_data",
                "description": "Fetch sales data for a given quarter and year.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "quarter": {"type": "integer"},
                        "year": {"type": "integer"}
                    },
                    "required": ["quarter", "year"]
                }
            },
            {
                "name": "analyze_sales",
                "description": "Analyze the loaded sales data and produce findings.",
                "parameters": {"type": "object", "properties": {}}
            },
            {
                "name": "report",
                "description": "Produce the final report from findings.",
                "parameters": {
                    "type": "object",
                    "properties": {"findings": {"type": "string"}},
                    "required": ["findings"]
                }
            }
        ],
        "required_steps": ["fetch_sales_data", "analyze_sales"],
        "terminal_tools": ["report"],
        "states": [
            {
                "completed_steps": [],
                "pending_steps": ["fetch_sales_data", "analyze_sales"],
                "valid": {"name": "fetch_sales_data", "arguments": {"quarter": 4, "year": 2024}},
                "premature_terminal": {"name": "report", "arguments": {"findings": "Revenue grew 23% YoY."}},
                "wrong_tool_semantic": {"name": "analyze_sales", "arguments": {}},
                "invalid_args_schema": {"name": "fetch_sales_data", "arguments": {"quarter": "Q4", "year": "twenty twenty four"}},
            },
            {
                "completed_steps": ["fetch_sales_data"],
                "pending_steps": ["analyze_sales"],
                "valid": {"name": "analyze_sales", "arguments": {}},
                "premature_terminal": {"name": "report", "arguments": {"findings": "Revenue grew 23% YoY."}},
            },
            {
                "completed_steps": ["fetch_sales_data", "analyze_sales"],
                "pending_steps": [],
                "valid": {"name": "report", "arguments": {"findings": "Revenue grew 23% YoY. Top product: Widget Pro. Weakest region: APAC."}},
            }
        ]
    },
    {
        "source": "forge_synthetic",
        "user_request": "Fetch 10 records and summarize them.",
        "tools": [
            {
                "name": "fetch",
                "description": "Fetch records. The count parameter must be a zero-padded 4-digit string.",
                "parameters": {
                    "type": "object",
                    "properties": {"count": {"type": "string"}},
                    "required": ["count"]
                }
            },
            {
                "name": "summarize",
                "description": "Summarize fetched content.",
                "parameters": {
                    "type": "object",
                    "properties": {"content": {"type": "string"}},
                    "required": ["content"]
                }
            }
        ],
        "required_steps": ["fetch"],
        "terminal_tools": ["summarize"],
        "states": [
            {
                "completed_steps": [],
                "pending_steps": ["fetch"],
                "valid": {"name": "fetch", "arguments": {"count": "0010"}},
                "invalid_args_schema": {"name": "fetch", "arguments": {"count": 10}},
                "premature_terminal": {"name": "summarize", "arguments": {"content": "Fetched 10 records."}},
            }
        ]
    }
]

def build_forge_synthetic_rows() -> List[VerifierRow]:
    rows = []
    for wf in FORGE_WORKFLOWS:
        for state in wf["states"]:
            base = dict(
                source=wf["source"],
                user_request=wf["user_request"],
                tools=wf["tools"],
                required_steps=wf["required_steps"],
                completed_steps=state["completed_steps"],
                pending_steps=state["pending_steps"],
                terminal_tools=wf["terminal_tools"],
            )
            if "valid" in state:
                rows.append(make_row(
                    label="valid",
                    candidate=state["valid"],
                    rank_score=1.0,
                    metadata={"generator": "forge_synthetic", "negative_type": None},
                    **base,
                ))
            for label in ["premature_terminal", "wrong_tool_semantic", "invalid_args_schema"]:
                if label in state:
                    rows.append(make_row(
                        label=label,
                        candidate=state[label],
                        rank_score=0.05 if label != "invalid_args_schema" else 0.15,
                        metadata={"generator": "forge_synthetic", "negative_type": label},
                        **base,
                    ))

            if state.get("pending_steps") and "valid" in state and wf["terminal_tools"]:
                term = next((t for t in wf["tools"] if t["name"] in wf["terminal_tools"]), None)
                if term:
                    candidate_batch = [
                        state["valid"],
                        {"name": term["name"], "arguments": {"message": "Done", "findings": "Done", "content": "Done"}}
                    ]
                    rows.append(make_row(
                        label="unsafe_parallel_batch",
                        candidate=candidate_batch,
                        rank_score=0.02,
                        metadata={"generator": "forge_synthetic", "negative_type": "unsafe_parallel_batch"},
                        **base,
                    ))
    return rows

forge_rows = build_forge_synthetic_rows()
print("Forge synthetic rows:", len(forge_rows))
print(forge_rows[0].input_text)

# Extra Forge-contract synthetic rows.
# Public function-calling datasets rarely include required-step, terminal-tool, or prerequisite concepts.
def build_guardrail_augmented_rows(n_per_label: int = 250) -> List[VerifierRow]:
    rng = random.Random(SEED)
    rows = []

    domains = [
        {
            "noun": "sales report",
            "fetch": "fetch_sales_data",
            "analyze": "analyze_sales",
            "terminal": "report",
            "required_args": {"quarter": 4, "year": 2024},
            "missing_user": "Generate the sales report.",
            "clear_user": "Generate a Q4 2024 sales report.",
            "terminal_arg": "findings",
        },
        {
            "noun": "audit",
            "fetch": "legacy_list_accounts",
            "analyze": "legacy_run_checks",
            "terminal": "legacy_submit_audit",
            "required_args": {"account_id": "ACC-12345"},
            "missing_user": "Run the audit for the account.",
            "clear_user": "Run the audit for account ACC-12345.",
            "terminal_arg": "report",
        },
        {
            "noun": "inventory summary",
            "fetch": "load_inventory",
            "analyze": "calculate_stock_risk",
            "terminal": "send_inventory_summary",
            "required_args": {"warehouse": "AUS-1"},
            "missing_user": "Send the inventory summary.",
            "clear_user": "Send the inventory summary for warehouse AUS-1.",
            "terminal_arg": "content",
        },
    ]

    for _ in range(n_per_label):
        d = rng.choice(domains)
        tools = [
            {
                "name": d["fetch"],
                "description": f"Load source data for the {d['noun']}.",
                "parameters": {
                    "type": "object",
                    "properties": {k: {"type": "string" if isinstance(v, str) else "integer"} for k, v in d["required_args"].items()},
                    "required": list(d["required_args"].keys()),
                },
            },
            {
                "name": d["analyze"],
                "description": f"Analyze loaded data for the {d['noun']}.",
                "parameters": {"type": "object", "properties": {}},
            },
            {
                "name": d["terminal"],
                "description": f"Submit the final {d['noun']}.",
                "parameters": {
                    "type": "object",
                    "properties": {d["terminal_arg"]: {"type": "string"}},
                    "required": [d["terminal_arg"]],
                },
            },
        ]

        required_steps = [d["fetch"], d["analyze"]]
        terminal_tools = [d["terminal"]]

        rows.append(make_row(
            source="forge_augmented",
            label="missing_prerequisite",
            user_request=d["clear_user"],
            tools=tools,
            candidate={"name": d["analyze"], "arguments": {}},
            rank_score=0.05,
            metadata={"generator": "forge_augmented", "negative_type": "missing_prerequisite"},
            required_steps=required_steps,
            completed_steps=[],
            pending_steps=required_steps,
            terminal_tools=terminal_tools,
        ))

        rows.append(make_row(
            source="forge_augmented",
            label="premature_terminal",
            user_request=d["clear_user"],
            tools=tools,
            candidate={"name": d["terminal"], "arguments": {d["terminal_arg"]: "Done."}},
            rank_score=0.05,
            metadata={"generator": "forge_augmented", "negative_type": "premature_terminal"},
            required_steps=required_steps,
            completed_steps=[],
            pending_steps=required_steps,
            terminal_tools=terminal_tools,
        ))

        rows.append(make_row(
            source="forge_augmented",
            label="unsafe_parallel_batch",
            user_request=d["clear_user"],
            tools=tools,
            candidate=[
                {"name": d["fetch"], "arguments": d["required_args"]},
                {"name": d["terminal"], "arguments": {d["terminal_arg"]: "Done."}},
            ],
            rank_score=0.02,
            metadata={"generator": "forge_augmented", "negative_type": "unsafe_parallel_batch"},
            required_steps=required_steps,
            completed_steps=[],
            pending_steps=required_steps,
            terminal_tools=terminal_tools,
        ))

        guessed_args = dict(d["required_args"])
        guessed_args[list(guessed_args.keys())[0]] = "UNKNOWN"
        rows.append(make_row(
            source="forge_augmented",
            label="needs_clarification",
            user_request=d["missing_user"],
            tools=tools,
            candidate={"name": d["fetch"], "arguments": guessed_args},
            rank_score=0.25,
            metadata={"generator": "forge_augmented", "negative_type": "needs_clarification"},
            required_steps=required_steps,
            completed_steps=[],
            pending_steps=required_steps,
            terminal_tools=terminal_tools,
        ))

    return rows

forge_rows = forge_rows + build_guardrail_augmented_rows(n_per_label=250)
print("Forge rows after augmentation:", len(forge_rows))
print(pd.Series([r.label for r in forge_rows]).value_counts())


## 5. Hard-negative generation for public datasets

For each valid gold call, generate negative candidates:

- wrong tool from same tool set
- unknown tool
- missing required args
- invalid arg shape
- no-tool/text answer when a tool is likely needed
- malformed tool call

This converts generation datasets into classifier/ranker datasets.


In [ ]:
def required_args_for_tool(tool: Dict[str, Any]) -> List[str]:
    params = tool.get("parameters") or {}
    req = params.get("required") or []
    return [str(x) for x in req] if isinstance(req, list) else []

def tool_by_name(tools: List[Dict[str, Any]], name: str) -> Optional[Dict[str, Any]]:
    for t in tools:
        if t.get("name") == name:
            return t
    return None

def wrong_tool_candidate(tools: List[Dict[str, Any]], gold_name: str, gold_args: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    others = [t for t in tools if t.get("name") != gold_name]
    if not others:
        return None
    t = random.choice(others)
    return {"name": t.get("name", "unknown_tool"), "arguments": gold_args or {}}

def missing_arg_candidate(tool: Dict[str, Any], call: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    req = required_args_for_tool(tool)
    if not req:
        return None
    args = dict(call.get("arguments") or {})
    removed = False
    for k in req:
        if k in args:
            args.pop(k)
            removed = True
            break
    if not removed:
        args = {}
    return {"name": call["name"], "arguments": args}

def invalid_arg_candidate(tool: Dict[str, Any], call: Dict[str, Any]) -> Dict[str, Any]:
    args = dict(call.get("arguments") or {})
    if not args:
        args = {"_unexpected": ["not", "a", "valid", "shape"]}
    else:
        k = random.choice(list(args.keys()))
        v = args[k]
        if isinstance(v, str):
            args[k] = {"bad": v}
        elif isinstance(v, (int, float)):
            args[k] = "not_a_number"
        elif isinstance(v, dict):
            args[k] = "not_an_object"
        else:
            args[k] = {"bad": "type"}
    return {"name": call["name"], "arguments": args}

def make_public_rows(examples: List[Dict[str, Any]], max_rows: Optional[int] = None) -> List[VerifierRow]:
    rows = []
    iter_examples = examples[:max_rows] if max_rows else examples

    for ex in tqdm(iter_examples, desc="building verifier rows"):
        source = ex["source"]
        user_request = ex["user_request"]
        tools = ex["tools"]
        gold_calls = ex["gold_calls"]

        required_steps = []
        completed_steps = []
        pending_steps = []
        terminal_tools = []

        for call in gold_calls[:3]:
            if not isinstance(call, dict) or not call.get("name"):
                continue
            tool = tool_by_name(tools, call["name"])
            if not tool:
                continue

            rows.append(make_row(
                source=source,
                label="valid",
                user_request=user_request,
                tools=tools,
                candidate=call,
                rank_score=1.0,
                metadata={"generator": "public_positive", "gold_tool": call["name"]},
                required_steps=required_steps,
                completed_steps=completed_steps,
                pending_steps=pending_steps,
                terminal_tools=terminal_tools,
            ))

            cand = wrong_tool_candidate(tools, call["name"], call.get("arguments") or {})
            if cand:
                rows.append(make_row(
                    source=source,
                    label="wrong_tool_semantic",
                    user_request=user_request,
                    tools=tools,
                    candidate=cand,
                    rank_score=0.10,
                    metadata={"generator": "hard_negative", "negative_type": "wrong_tool", "gold_tool": call["name"]},
                    required_steps=required_steps,
                    completed_steps=completed_steps,
                    pending_steps=pending_steps,
                    terminal_tools=terminal_tools,
                ))

            cand = missing_arg_candidate(tool, call)
            if cand:
                rows.append(make_row(
                    source=source,
                    label="missing_required_args",
                    user_request=user_request,
                    tools=tools,
                    candidate=cand,
                    rank_score=0.20,
                    metadata={"generator": "hard_negative", "negative_type": "missing_required_args", "gold_tool": call["name"]},
                    required_steps=required_steps,
                    completed_steps=completed_steps,
                    pending_steps=pending_steps,
                    terminal_tools=terminal_tools,
                ))

            cand = invalid_arg_candidate(tool, call)
            rows.append(make_row(
                source=source,
                label="invalid_args_schema",
                user_request=user_request,
                tools=tools,
                candidate=cand,
                rank_score=0.20,
                metadata={"generator": "hard_negative", "negative_type": "invalid_args_schema", "gold_tool": call["name"]},
                required_steps=required_steps,
                completed_steps=completed_steps,
                pending_steps=pending_steps,
                terminal_tools=terminal_tools,
            ))

            rows.append(make_row(
                source=source,
                label="unknown_tool",
                user_request=user_request,
                tools=tools,
                candidate={"name": "definitely_not_a_real_tool", "arguments": call.get("arguments") or {}},
                rank_score=0.0,
                metadata={"generator": "hard_negative", "negative_type": "unknown_tool", "gold_tool": call["name"]},
                required_steps=required_steps,
                completed_steps=completed_steps,
                pending_steps=pending_steps,
                terminal_tools=terminal_tools,
            ))

            rows.append(make_row(
                source=source,
                label="malformed_tool_call",
                user_request=user_request,
                tools=tools,
                candidate="{not valid json",
                rank_score=0.0,
                metadata={"generator": "hard_negative", "negative_type": "malformed_tool_call", "gold_tool": call["name"]},
                required_steps=required_steps,
                completed_steps=completed_steps,
                pending_steps=pending_steps,
                terminal_tools=terminal_tools,
            ))

            if random.random() < 0.20:
                rows.append(make_row(
                    source=source,
                    label="tool_not_needed",
                    user_request=user_request,
                    tools=tools,
                    candidate={"text_response": "I can answer this directly without tools."},
                    rank_score=0.30,
                    metadata={"generator": "hard_negative", "negative_type": "text_instead_of_tool", "gold_tool": call["name"]},
                    required_steps=required_steps,
                    completed_steps=completed_steps,
                    pending_steps=pending_steps,
                    terminal_tools=terminal_tools,
                ))

    return rows

public_rows = make_public_rows(normalized_examples)
all_rows = public_rows + forge_rows
print("public rows:", len(public_rows))
print("all rows:", len(all_rows))


In [ ]:
#@title Balance classes lightly
def rows_to_df(rows: List[VerifierRow]) -> pd.DataFrame:
    return pd.DataFrame([asdict(r) for r in rows])

df = rows_to_df(all_rows)
if df.empty:
    raise RuntimeError("No training rows were created. Inspect dataset schemas or enable Forge synthetic rows.")

# Preserve raw label and apply selected label mode.
df["raw_label"] = df["label"]
df["label"] = df["label"].map(normalize_label)

print("Raw labels:")
print(df["raw_label"].value_counts(dropna=False))
print("\nTraining labels:")
print(df["label"].value_counts(dropna=False))

MIN_USEFUL_ROWS = 100
if len(df) < MIN_USEFUL_ROWS:
    print(
        f"WARNING: only {len(df)} rows were created. This is fine for testing notebook mechanics, "
        "but not enough to train a useful classifier. Check public dataset parsing/auth."
    )

# Cap very common labels and gently oversample small-but-present labels.
# Do not overinterpret metrics for classes that are mostly synthetic.
MAX_PER_LABEL = 20_000
MIN_PER_LABEL = 500 if LABEL_MODE == "diagnostic" else 1_000

pieces = []
for label, group in df.groupby("label", sort=False):
    n = min(MAX_PER_LABEL, max(len(group), MIN_PER_LABEL))
    replace = len(group) < n
    pieces.append(group.sample(n=n, replace=replace, random_state=SEED))

balanced = (
    pd.concat(pieces, ignore_index=True)
      .sample(frac=1.0, random_state=SEED)
      .reset_index(drop=True)
)

print("\nBalanced training labels:")
print(balanced["label"].value_counts())
balanced.head()

In [ ]:
#@title Write JSONL dataset and train/valid/test splits
dataset_path = DATA_DIR / "toolcall_verifier_dataset.jsonl"
with dataset_path.open("w") as f:
    for row in balanced.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def can_stratify(frame: pd.DataFrame, label_col: str, test_size: float) -> bool:
    if frame.empty or frame[label_col].nunique() <= 1:
        return False
    counts = frame[label_col].value_counts()
    n_classes = len(counts)
    n_test = math.ceil(len(frame) * test_size)
    n_train = len(frame) - n_test
    return counts.min() >= 2 and n_test >= n_classes and n_train >= n_classes

def safe_train_valid_test_split(
    frame: pd.DataFrame,
    label_col: str = "label",
    test_size: float = 0.20,
    valid_fraction_of_temp: float = 0.50,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Stratifies when safe. Rare classes are kept in train so scikit-learn does not fail.
    This is for notebook robustness; for real metrics, collect enough rows per class.
    """
    frame = frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    counts = frame[label_col].value_counts()

    # Need enough examples for a two-stage split. Keep tiny classes in train.
    rare_labels = set(counts[counts < 3].index)
    rare_df = frame[frame[label_col].isin(rare_labels)]
    common_df = frame[~frame[label_col].isin(rare_labels)]

    if rare_labels:
        print(f"Keeping rare labels only in train for this split: {sorted(rare_labels)}")

    if len(common_df) < 3:
        train_df = frame.copy()
        valid_df = frame.sample(n=min(len(frame), max(1, len(frame) // 5)), random_state=SEED).copy()
        test_df = valid_df.copy()
        print("WARNING: very small dataset. Reusing sampled train rows for valid/test smoke checks.")
        return train_df, valid_df, test_df

    stratify_first = common_df[label_col] if can_stratify(common_df, label_col, test_size) else None
    train_common, tmp_df = train_test_split(
        common_df,
        test_size=test_size,
        random_state=SEED,
        stratify=stratify_first,
    )

    train_df = pd.concat([train_common, rare_df], ignore_index=True).sample(frac=1.0, random_state=SEED)

    if len(tmp_df) < 2:
        valid_df = train_df.sample(n=min(len(train_df), 8), random_state=SEED).copy()
        test_df = valid_df.copy()
        print("WARNING: temp split too small. Reusing sampled train rows for valid/test smoke checks.")
        return train_df, valid_df, test_df

    stratify_second = tmp_df[label_col] if can_stratify(tmp_df, label_col, valid_fraction_of_temp) else None
    valid_df, test_df = train_test_split(
        tmp_df,
        test_size=valid_fraction_of_temp,
        random_state=SEED,
        stratify=stratify_second,
    )

    return (
        train_df.reset_index(drop=True),
        valid_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

train_df, valid_df, test_df = safe_train_valid_test_split(balanced)

for name, split_df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    path = DATA_DIR / f"{name}.jsonl"
    with path.open("w") as f:
        for row in split_df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(name, len(split_df), path)
    print(split_df["label"].value_counts().to_string())

print("Dataset written:", dataset_path)


## 6. Load JSONL splits with Hugging Face Datasets


In [ ]:
ds = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "train.jsonl"),
        "validation": str(DATA_DIR / "valid.jsonl"),
        "test": str(DATA_DIR / "test.jsonl"),
    }
)

def add_label_id(ex):
    # Rows are already normalized before writing splits, but keep this defensive.
    ex["label"] = normalize_label(ex["label"])
    ex["labels"] = label2id[ex["label"]]
    return ex

ds = ds.map(add_label_id)
ds

## 7. Tokenize and train a sequence classifier

This is a cross-encoder-style classifier: the workflow state, available tools, and candidate call are serialized into one input.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Optional token-length diagnostic. If p90/p95 are far below MAX_LENGTH, lower MAX_LENGTH for faster training.
def estimate_token_lengths(dataset, n: int = 2000):
    sample = dataset.select(range(min(n, len(dataset))))
    lengths = []
    for row in sample:
        lengths.append(len(tokenizer(row["input_text"], truncation=False)["input_ids"]))
    return pd.Series(lengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

print("Token length sample:")
print(estimate_token_lengths(ds["train"]))

remove_cols = [c for c in ds["train"].column_names if c not in ("labels",)]
tokenized = ds.map(tokenize_batch, batched=True, remove_columns=remove_cols)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if USE_CUDA else None,
)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

ALL_LABEL_IDS = list(range(len(LABELS)))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    present_precision, present_recall, present_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    full_precision, full_recall, full_f1, _ = precision_recall_fscore_support(
        labels, preds, labels=ALL_LABEL_IDS, average="macro", zero_division=0
    )

    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "macro_precision": present_precision,
        "macro_recall": present_recall,
        "macro_f1": present_f1,
        "macro_precision_all_labels": full_precision,
        "macro_recall_all_labels": full_recall,
        "macro_f1_all_labels": full_f1,
    }

def build_class_weights(train_frame: pd.DataFrame) -> Optional[torch.Tensor]:
    if not USE_CLASS_WEIGHTS:
        return None

    counts = train_frame["label"].value_counts()
    weights = []
    for label in LABELS:
        count = counts.get(label, 0)
        if count <= 0:
            weights.append(0.0)
        else:
            raw = len(train_frame) / (len(LABELS) * count)
            weights.append(min(float(raw), MAX_CLASS_WEIGHT))

    weights = np.array(weights, dtype=np.float32)
    nonzero = weights[weights > 0]
    if len(nonzero):
        weights[weights > 0] = weights[weights > 0] / nonzero.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights: Optional[torch.Tensor] = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is None:
            loss = outputs.get("loss")
        else:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

class_weights = build_class_weights(train_df)
if class_weights is not None:
    print("Class weights:")
    for label, weight in zip(LABELS, class_weights.tolist()):
        print(f"  {label:28s} {weight:.3f}")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=USE_FP16,
    bf16=USE_BF16,
    tf32=USE_CUDA,

    auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
    dataloader_num_workers=2 if USE_CUDA else 0,
    group_by_length=GROUP_BY_LENGTH,
    save_total_limit=2,

    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
#@title Evaluate on test set
metrics = trainer.evaluate(tokenized["test"])
print(metrics)

pred_out = trainer.predict(tokenized["test"])
preds = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids
ALL_LABEL_IDS = list(range(len(LABELS)))

# classification_report needs explicit labels when the test split does not contain every class.
report = classification_report(
    labels,
    preds,
    labels=ALL_LABEL_IDS,
    target_names=LABELS,
    zero_division=0,
)
print(report)

cm = confusion_matrix(labels, preds, labels=ALL_LABEL_IDS)
cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
display(cm_df)

# Inspect high-confidence mistakes. These are usually the fastest way to find label noise.
probs = torch.softmax(torch.tensor(pred_out.predictions), dim=-1).numpy()
conf = probs.max(axis=-1)
pred_labels = [id2label[int(i)] for i in preds]
true_labels = [id2label[int(i)] for i in labels]

test_raw = ds["test"].to_pandas()
test_raw["true_label"] = true_labels
test_raw["pred_label"] = pred_labels
test_raw["confidence"] = conf
test_raw["correct"] = test_raw["true_label"] == test_raw["pred_label"]

mistakes = (
    test_raw[test_raw["correct"] == False]
    .sort_values("confidence", ascending=False)
    [["true_label", "pred_label", "confidence", "source", "input_text", "metadata"]]
)

print("High-confidence mistakes:")
display(mistakes.head(25))

In [ ]:
#@title Save model and tokenizer
final_model_dir = MODEL_DIR / "final"
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))
(DATA_DIR / "training_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved to:", final_model_dir)


## 8. Quick inference helper

Use this to test candidate calls and estimate latency.


In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=str(final_model_dir),
    tokenizer=str(final_model_dir),
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)

def score_candidate(
    user_request: str,
    tools: List[Dict[str, Any]],
    candidate: Any,
    required_steps=None,
    completed_steps=None,
    pending_steps=None,
    terminal_tools=None,
):
    text = serialize_state(
        user_request=user_request,
        tools=tools,
        candidate=candidate,
        required_steps=required_steps,
        completed_steps=completed_steps,
        pending_steps=pending_steps,
        terminal_tools=terminal_tools,
    )
    start = time.perf_counter()
    scores = clf(text, truncation=True, max_length=MAX_LENGTH)[0]
    elapsed_ms = (time.perf_counter() - start) * 1000
    scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return scores[:5], elapsed_ms

sample = FORGE_WORKFLOWS[0]
scores, elapsed_ms = score_candidate(
    user_request=sample["user_request"],
    tools=sample["tools"],
    candidate={"name": "report", "arguments": {"findings": "Done"}},
    required_steps=sample["required_steps"],
    completed_steps=[],
    pending_steps=sample["required_steps"],
    terminal_tools=sample["terminal_tools"],
)
print("Latency ms:", round(elapsed_ms, 2))
scores


## 9. Optional ONNX export

The Rust side should load:

- `model.onnx`
- tokenizer files
- `labels.json`
- `thresholds.json`

Start with shadow mode before enforcement.


In [ ]:
#@title Export to ONNX with Optimum
# This can take a few minutes.
!python -m optimum.exporters.onnx \
  --model "{final_model_dir}" \
  --task text-classification \
  "{ONNX_DIR}"

import shutil
shutil.copy(DATA_DIR / "labels.json", ONNX_DIR / "labels.json")

thresholds = {
    "mode": "shadow",
    "enforce_min_confidence": 0.98,
    "advisory_min_confidence": 0.80,
    "notes": "Deterministic guardrails remain authoritative. Use ML as scorer/ranker first."
}
(ONNX_DIR / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
print("ONNX artifacts:", list(ONNX_DIR.iterdir()))


In [ ]:
#@title Optional dynamic quantization
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_files = list(ONNX_DIR.glob("*.onnx"))
if onnx_files:
    src = onnx_files[0]
    dst = ONNX_DIR / "model_quantized.onnx"
    quantize_dynamic(str(src), str(dst), weight_type=QuantType.QInt8)
    print("Quantized:", dst)
else:
    print("No ONNX file found.")


## 10. Package artifacts


In [ ]:
#@title Zip dataset, model, and ONNX artifacts
import shutil

artifact_root = WORKDIR / "artifacts"
artifact_root.mkdir(exist_ok=True)

zip_path = shutil.make_archive(
    base_name=str(artifact_root / "toolcall_verifier_artifacts"),
    format="zip",
    root_dir=str(WORKDIR),
    base_dir="."
)
print("Wrote:", zip_path)


## 11. Optional upload to Hugging Face Hub

Suggested repos:

- dataset: `YOUR_USERNAME/toolcall-verifier-dataset`
- model: `YOUR_USERNAME/toolcall-verifier-classifier`

Keep this private until you verify dataset licensing and remove any proprietary Forge traces.


In [ ]:
#@title Optional: upload dataset/model to Hub
UPLOAD_TO_HUB = False  #@param {type:"boolean"}
HF_DATASET_REPO = "cowWhySo/toolcall-verifier-dataset"  #@param {type:"string"}
HF_MODEL_REPO = "cowWhySo/toolcall-verifier-classifier"  #@param {type:"string"}
PRIVATE = True  #@param {type:"boolean"}

if UPLOAD_TO_HUB:
    from huggingface_hub import HfApi, create_repo, upload_folder
    api = HfApi()
    create_repo(HF_DATASET_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True)
    upload_folder(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        folder_path=str(DATA_DIR),
        commit_message="Add tool-call verifier dataset",
    )

    create_repo(HF_MODEL_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
    upload_folder(
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        folder_path=str(final_model_dir),
        commit_message="Add tool-call verifier classifier",
    )
    print("Uploaded dataset and model.")
else:
    print("Upload disabled.")


## 12. Training and labelling guidance

Recommended workflow:

1. Start with `LABEL_MODE = "diagnostic"` while building the dataset. This keeps every violation visible.
2. Use the confusion matrix and high-confidence mistakes table to identify noisy labels.
3. Once deterministic Rust guardrails are strong, run a second model with `LABEL_MODE = "production"`.
4. In production mode, deterministic failures collapse into `deterministic_invalid`, while the model focuses on `valid`, `wrong_tool_semantic`, `tool_not_needed`, and `needs_clarification`.

Labelling rules:

- Use deterministic validators for `unknown_tool`, `malformed_tool_call`, `invalid_args_schema`, `missing_required_args`, `premature_terminal`, `missing_prerequisite`, and `unsafe_parallel_batch`.
- Use ML mainly for `wrong_tool_semantic`, `tool_not_needed`, `needs_clarification`, and next-tool ranking.
- Do not let the ML model silently rewrite arguments.
- Keep the model in shadow mode until the valid-call false-block rate is very low.

Training guidance:

- On T4, prefer `MAX_LENGTH = 512`, `TRAIN_BATCH_SIZE = 32`, and fp16.
- On L4/A100, try `TRAIN_BATCH_SIZE = 64` and bf16.
- If token-length p95 is below 512, do not train at 768.
- If rare Forge classes are mostly synthetic, report public-dataset metrics separately from Forge-contract metrics.

## 12. Rust integration sketch

Recommended production order:

1. deterministic parser/schema checks
2. step/prerequisite/terminal enforcement
3. ML scorer only for valid-looking ambiguous calls
4. shadow logging
5. advisory nudges
6. high-confidence enforcement only after eval proof

Suggested Rust API:

```rust
pub trait ToolCallScorer {
    fn score(&self, ctx: &ScoringContext, candidate: &ToolCall) -> ToolCallScore;
}

pub struct ToolCallScore {
    pub label: ToolCallClass,
    pub confidence: f32,
    pub rank_score: f32,
}
```

Use model output for:

- `wrong_tool_semantic`
- next-tool ranking
- nudge selection
- recovery-mode triage

Do not use model output for:

- overriding deterministic safety
- executing tools
- accepting malformed calls
- silently rewriting arguments
